# TD  Construction de la collection `entreprise`

On dispose de l'export KBO Open Data (8 fichiers CSV) monte sur le volume Docker,
et d'une base MongoDB vide. L'objectif de ce TD est de construire, fonction par
fonction, le pipeline qui charge ces CSV dans MongoDB puis les joint pour produire
une collection `entreprise` : un document par entreprise, avec ses etablissements
et ses branches imbriques.



In [1]:
!pip install pymongo

     |████████████████████████████████| 763 kB 6.3 MB/s eta 0:00:01
     |████████████████████████████████| 313 kB 4.8 MB/s eta 0:00:01
You should consider upgrading via the '/Users/theo-dev/Dev/M2_IPSSI/M2_BIGDATA/.venv/bin/python3 -m pip install --upgrade pip' command.


In [10]:
import os
import csv
from pathlib import Path
import pymongo

# En local, le dossier data est dans le même répertoire que le notebook
DATA_DIR = Path("./data")

# En local, on attaque MongoDB sur "localhost" (le port 27017 est exposé par Docker)
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "kbo_db"

client = pymongo.MongoClient(MONGO_URI)
db = client[DB_NAME]
print("Connexion à MongoDB réussie !")

Connexion à MongoDB réussie !


## 1) Chargement d'un CSV dans une collection

Les fichiers de l'export sont volumineux. 

Ecrivez une fonction qui charge un fichier CSV donne dans une
collection MongoDB donnee par batch

Appliquez cette fonction aux 8 fichiers de l'export pour peupler 8 collections,
une par fichier.


In [12]:
def load_csv_to_mongo(file_path: Path, collection_name: str, batch_size: int = 10000):
    collection = db[collection_name]
    collection.drop() # On réinitialise la collection
    
    batch = []
    with open(file_path, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            batch.append(row)
            if len(batch) >= batch_size:
                collection.insert_many(batch)
                batch = []
        if batch:
            collection.insert_many(batch)
    
    print(f"Collection '{collection_name}' chargée.")

# Mapping des fichiers visibles dans ton arborescence vers les collections
files_and_collections = {
    "enterprise.csv": "enterprises",
    "establishment.csv": "establishments",
    "branch.csv": "branches",
    "denomination.csv": "denominations",
    "address.csv": "addresses",
    "contact.csv": "contacts",
    "activity.csv": "activities"
}

for filename, coll_name in files_and_collections.items():
    file_path = DATA_DIR / filename
    if file_path.exists():
        load_csv_to_mongo(file_path, coll_name)
    else:
        print(f"Fichier non trouvé : {file_path}")

Collection 'enterprises' chargée.
Collection 'establishments' chargée.
Collection 'branches' chargée.
Collection 'denominations' chargée.
Collection 'addresses' chargée.
Collection 'contacts' chargée.
Collection 'activities' chargée.


## 2) Le schema de la jointure

indexez dans chaque collection concernee le ou les champs qui serviront de cle de jointure.


In [13]:
# Indexation des clés pour optimiser les jointures ($lookup)
for coll in ['denominations', 'addresses', 'contacts', 'activities']:
    db[coll].create_index("EntityNumber")

db['establishments'].create_index("EnterpriseNumber")
db['branches'].create_index("EnterpriseNumber")
db['enterprises'].create_index("EnterpriseNumber", unique=True)

print("Index créés avec succès.")

Index créés avec succès.


## 3) Rejoindre les details d'une entite

Trois niveaux d'entites : entreprise, etablissement, branche -- ont chacun
besoin des memes quatre informations complementaires : leurs denominations,
leurs adresses, leurs contacts et leurs activites. Ces quatre informations
vivent chacune dans leur propre collection, et s'y rattachent toujours de la
meme maniere, quel que soit le type d'entite concerne.

Ecrivez une fonction reutilisable qui, etant donne le nom du champ a utiliser
comme cle de jointure du cote de l'entite courante, construit les etapes
d'agregation necessaires pour rattacher ces quatre informations. Cette fonction
sera appelee trois fois dans la suite du TD, une fois par niveau d'entite, avec
a chaque fois un nom de champ different.


In [14]:

# Indexation des clés pour optimiser les jointures ($lookup)
for coll in ['denominations', 'addresses', 'contacts', 'activities']:
    db[coll].create_index("EntityNumber")

db['establishments'].create_index("EnterpriseNumber")
db['branches'].create_index("EnterpriseNumber")
db['enterprises'].create_index("EnterpriseNumber", unique=True)

print("Index créés avec succès.")


def _detail_lookups(primary_key: str):
    """Génère les étapes d'agrégation pour joindre les 4 informations de base."""
    return [
        {
            "$lookup": {
                "from": coll,
                "localField": primary_key,
                "foreignField": "EntityNumber",
                "as": coll
            }
        }
        for coll in ['denominations', 'addresses', 'contacts', 'activities']
    ]

Index créés avec succès.


## 4) Rejoindre les succursales

Une succursale represente la presence en Belgique d'une entreprise etrangere.
Chaque succursale est rattachee a une entreprise, et a, comme vu a la question
3, ses propres denominations et adresses (mais jamais de contacts ni
d'activites).

Ecrivez une fonction qui construit l'etape d'agregation permettant de rattacher,
pour chaque entreprise, la liste de ses succursales completes, chaque
succursale devant elle-meme deja porter ses propres denominations et adresses,
obtenues via la fonction de la question 3. Le lien entre une entreprise et ses
succursales ne se fait pas sur le meme champ que celui utilise a l'interieur
d'une succursale pour aller chercher ses propres denominations et adresses :
il faudra donc correler explicitement les deux niveaux.


In [15]:
def get_branches_lookup():
    return {
        "$lookup": {
            "from": "branches",
            "localField": "EnterpriseNumber",
            "foreignField": "EnterpriseNumber",
            "pipeline": _detail_lookups("Id"), # La clé interne d'une succursale est 'Id'
            "as": "branches"
        }
    }

def get_establishments_lookup():
    return {
        "$lookup": {
            "from": "establishments",
            "localField": "EnterpriseNumber",
            "foreignField": "EnterpriseNumber",
            "pipeline": _detail_lookups("EstablishmentNumber"),
            "as": "establishments"
        }
    }

## 5) Rejoindre les etablissements

Meme exercice que la question 4, mais pour les etablissements, les unites
operationnelles d'une entreprise belge. 

Comme pour les succursales, chaque etablissement doit deja porter ses propres denominations, adresses, contacts et
activites (obtenus via la fonction de la question 3) avant d'etre rattache a son entreprise.


In [16]:
import pprint

pipeline = []
pipeline.extend(_detail_lookups("EnterpriseNumber"))
pipeline.append(get_establishments_lookup())
pipeline.append(get_branches_lookup())
pipeline.append({"$out": "entreprise"}) # Écriture dans la collection finale

print("Exécution du pipeline (cela peut prendre un moment)...")
db['enterprises'].aggregate(pipeline, allowDiskUse=True)
print("Collection 'entreprise' construite !")

# --- Vérifications ---
print("\n--- Exemple d'entreprise avec établissement ---")
pprint.pprint(
    db['entreprise'].find_one({"establishments.0": {"$exists": True}})
)

print("\n--- Exemple d'entreprise avec succursale ---")
pprint.pprint(
    db['entreprise'].find_one({"branches.0": {"$exists": True}})
)

Exécution du pipeline (cela peut prendre un moment)...
Collection 'entreprise' construite !

--- Exemple d'entreprise avec établissement ---
{'EnterpriseNumber': '0200.065.765',
 'JuridicalForm': '416',
 'JuridicalFormCAC': '',
 'JuridicalSituation': '000',
 'StartDate': '09-08-1960',
 'Status': 'AC',
 'TypeOfEnterprise': '2',
 '_id': ObjectId('6a676bc449d1f33a2397b64d'),
 'activities': [{'ActivityGroup': '006',
                 'Classification': 'MAIN',
                 'EntityNumber': '0200.065.765',
                 'NaceCode': '84130',
                 'NaceVersion': '2025',
                 '_id': ObjectId('6a676c1a49d1f33a23398269')},
                {'ActivityGroup': '001',
                 'Classification': 'MAIN',
                 'EntityNumber': '0200.065.765',
                 'NaceCode': '68121',
                 'NaceVersion': '2025',
                 '_id': ObjectId('6a676c1a49d1f33a2339826a')},
                {'ActivityGroup': '001',
                 'Classification': '

## 6) Assembler et executer le pipeline complet

Combinez les fonctions precedentes en un seul pipeline d'agregation, lance sur
la collection contenant les entreprises : les quatre informations
complementaires de l'entreprise elle-meme, puis ses etablissements, puis ses
succursales.

le resultat de ce pipeline doit s'ecrire dans une nouvelle collection,

Executez le pipeline, et afficher le document
complet d'une entreprise qui possede au moins un etablissement, et d'une
entreprise qui possede au moins une succursale.


In [18]:
{
  "_id": "0403.449.823",
  "EnterpriseNumber": "0403.449.823",
  "Status": "AC",
  "JuridicalSituation": "000",
  "TypeOfEnterprise": "2",
  "JuridicalForm": "014",
  "JuridicalFormCAC": "",
  "StartDate": "01-01-1968",
  "denominations": [
    {
      "_id": {
        "$oid": "6a66155ee03570e5a50ec781"
      },
      "EntityNumber": "0403.449.823",
      "Language": "2",
      "TypeOfDenomination": "001",
      "Denomination": "Thornton"
    }
  ],
  "addresses": [
    {
      "_id": {
        "$oid": "6a6615a0e03570e5a541ea5e"
      },
      "EntityNumber": "0403.449.823",
      "TypeOfAddress": "REGO",
      "CountryNL": "",
      "CountryFR": "",
      "Zipcode": "2030",
      "MunicipalityNL": "Antwerpen",
      "MunicipalityFR": "Antwerpen",
      "StreetNL": "Treurenborg",
      "StreetFR": "Treurenborg",
      "HouseNumber": "9",
      "Box": "",
      "ExtraAddressInfo": "",
      "DateStrikingOff": ""
    }
  ],
  "contacts": [],
  "activities": [
    {
      "_id": {
        "$oid": "6a661620e03570e5a579000b"
      },
      "EntityNumber": "0403.449.823",
      "ActivityGroup": "006",
      "NaceVersion": "2025",
      "NaceCode": "33110",
      "Classification": "MAIN"
    },
    {
      "_id": {
        "$oid": "6a661620e03570e5a579000c"
      },
      "EntityNumber": "0403.449.823",
      "ActivityGroup": "006",
      "NaceVersion": "2003",
      "NaceCode": "34201",
      "Classification": "MAIN"
    },
    {
      "_id": {
        "$oid": "6a661620e03570e5a579000d"
      },
      "EntityNumber": "0403.449.823",
      "ActivityGroup": "001",
      "NaceVersion": "2025",
      "NaceCode": "33110",
      "Classification": "MAIN"
    },
    {
      "_id": {
        "$oid": "6a661620e03570e5a579000e"
      },
      "EntityNumber": "0403.449.823",
      "ActivityGroup": "001",
      "NaceVersion": "2025",
      "NaceCode": "77399",
      "Classification": "SECO"
    },
    {
      "_id": {
        "$oid": "6a661620e03570e5a579000f"
      },
      "EntityNumber": "0403.449.823",
      "ActivityGroup": "001",
      "NaceVersion": "2025",
      "NaceCode": "46649",
      "Classification": "SECO"
    },
    {
      "_id": {
        "$oid": "6a661620e03570e5a5790010"
      },
      "EntityNumber": "0403.449.823",
      "ActivityGroup": "001",
      "NaceVersion": "2025",
      "NaceCode": "46841",
      "Classification": "SECO"
    },
    {
      "_id": {
        "$oid": "6a661620e03570e5a5790011"
      },
      "EntityNumber": "0403.449.823",
      "ActivityGroup": "006",
      "NaceVersion": "2008",
      "NaceCode": "33110",
      "Classification": "MAIN"
    },
    {
      "_id": {
        "$oid": "6a661620e03570e5a5790012"
      },
      "EntityNumber": "0403.449.823",
      "ActivityGroup": "001",
      "NaceVersion": "2008",
      "NaceCode": "77399",
      "Classification": "SECO"
    },
    {
      "_id": {
        "$oid": "6a661620e03570e5a5790013"
      },
      "EntityNumber": "0403.449.823",
      "ActivityGroup": "001",
      "NaceVersion": "2008",
      "NaceCode": "46741",
      "Classification": "SECO"
    },
    {
      "_id": {
        "$oid": "6a661620e03570e5a5790014"
      },
      "EntityNumber": "0403.449.823",
      "ActivityGroup": "001",
      "NaceVersion": "2008",
      "NaceCode": "46699",
      "Classification": "SECO"
    },
    {
      "_id": {
        "$oid": "6a661620e03570e5a5790015"
      },
      "EntityNumber": "0403.449.823",
      "ActivityGroup": "001",
      "NaceVersion": "2008",
      "NaceCode": "33110",
      "Classification": "MAIN"
    }
  ],
  "establishments": [
    {
      "_id": {
        "$oid": "6a66153be03570e5a5f49aba"
      },
      "EstablishmentNumber": "2.000.000.339",
      "StartDate": "01-11-1974",
      "EnterpriseNumber": "0403.449.823",
      "denominations": [
        {
          "_id": {
            "$oid": "6a661589e03570e5a52e27a4"
          },
          "EntityNumber": "2.000.000.339",
          "Language": "2",
          "TypeOfDenomination": "003",
          "Denomination": "I.C.T.C."
        }
      ],
      "addresses": [
        {
          "_id": {
            "$oid": "6a6615d2e03570e5a553d4a2"
          },
          "EntityNumber": "2.000.000.339",
          "TypeOfAddress": "BAET",
          "CountryNL": "",
          "CountryFR": "",
          "Zipcode": "2030",
          "MunicipalityNL": "Antwerpen",
          "MunicipalityFR": "Antwerpen",
          "StreetNL": "Treurenborg",
          "StreetFR": "Treurenborg",
          "HouseNumber": "7",
          "Box": "",
          "ExtraAddressInfo": "",
          "DateStrikingOff": ""
        }
      ],
      "contacts": [],
      "activities": [
        {
          "_id": {
            "$oid": "6a6616b1e03570e5a5e09082"
          },
          "EntityNumber": "2.000.000.339",
          "ActivityGroup": "006",
          "NaceVersion": "2025",
          "NaceCode": "33110",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6616b1e03570e5a5e09083"
          },
          "EntityNumber": "2.000.000.339",
          "ActivityGroup": "006",
          "NaceVersion": "2003",
          "NaceCode": "34201",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6616b1e03570e5a5e09084"
          },
          "EntityNumber": "2.000.000.339",
          "ActivityGroup": "006",
          "NaceVersion": "2008",
          "NaceCode": "33110",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6616b1e03570e5a5e09085"
          },
          "EntityNumber": "2.000.000.339",
          "ActivityGroup": "003",
          "NaceVersion": "2008",
          "NaceCode": "52241",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6616b1e03570e5a5e09086"
          },
          "EntityNumber": "2.000.000.339",
          "ActivityGroup": "003",
          "NaceVersion": "2008",
          "NaceCode": "25999",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6616b1e03570e5a5e09087"
          },
          "EntityNumber": "2.000.000.339",
          "ActivityGroup": "003",
          "NaceVersion": "2003",
          "NaceCode": "63111",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6616b1e03570e5a5e09088"
          },
          "EntityNumber": "2.000.000.339",
          "ActivityGroup": "003",
          "NaceVersion": "2003",
          "NaceCode": "28755",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6616b1e03570e5a5e09089"
          },
          "EntityNumber": "2.000.000.339",
          "ActivityGroup": "003",
          "NaceVersion": "2025",
          "NaceCode": "25999",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6616b1e03570e5a5e0908a"
          },
          "EntityNumber": "2.000.000.339",
          "ActivityGroup": "003",
          "NaceVersion": "2025",
          "NaceCode": "52241",
          "Classification": "MAIN"
        }
      ]
    },
    {
      "_id": {
        "$oid": "6a66154ae03570e5a5018bfb"
      },
      "EstablishmentNumber": "2.272.310.221",
      "StartDate": "01-01-2018",
      "EnterpriseNumber": "0403.449.823",
      "denominations": [
        {
          "_id": {
            "$oid": "6a661594e03570e5a53713bb"
          },
          "EntityNumber": "2.272.310.221",
          "Language": "2",
          "TypeOfDenomination": "003",
          "Denomination": "I.C.T.C."
        }
      ],
      "addresses": [
        {
          "_id": {
            "$oid": "6a6615efe03570e5a560c5e3"
          },
          "EntityNumber": "2.272.310.221",
          "TypeOfAddress": "BAET",
          "CountryNL": "",
          "CountryFR": "",
          "Zipcode": "2030",
          "MunicipalityNL": "Antwerpen",
          "MunicipalityFR": "Antwerpen",
          "StreetNL": "Treurenborg",
          "StreetFR": "Treurenborg",
          "HouseNumber": "9",
          "Box": "",
          "ExtraAddressInfo": "",
          "DateStrikingOff": ""
        }
      ],
      "contacts": [],
      "activities": [
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f8fd"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "006",
          "NaceVersion": "2025",
          "NaceCode": "33110",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f8fe"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "006",
          "NaceVersion": "2008",
          "NaceCode": "33110",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f8ff"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2008",
          "NaceCode": "52241",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f900"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2008",
          "NaceCode": "46620",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f901"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2008",
          "NaceCode": "46699",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f902"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2008",
          "NaceCode": "77399",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f903"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2008",
          "NaceCode": "52100",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f904"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2008",
          "NaceCode": "46741",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f905"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2025",
          "NaceCode": "77399",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f906"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2025",
          "NaceCode": "46649",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f907"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2025",
          "NaceCode": "52100",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f908"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2025",
          "NaceCode": "46841",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f909"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2025",
          "NaceCode": "46620",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6617b7e03570e5a5c1f90a"
          },
          "EntityNumber": "2.272.310.221",
          "ActivityGroup": "003",
          "NaceVersion": "2025",
          "NaceCode": "52241",
          "Classification": "MAIN"
        }
      ]
    }
  ],
  "branches": []
}

{
  "_id": "0257.883.408",
  "EnterpriseNumber": "0257.883.408",
  "Status": "AC",
  "JuridicalSituation": "000",
  "TypeOfEnterprise": "2",
  "JuridicalForm": "030",
  "JuridicalFormCAC": "",
  "StartDate": "01-09-1995",
  "denominations": [
    {
      "_id": {
        "$oid": "6a66155ee03570e5a50e9ef2"
      },
      "EntityNumber": "0257.883.408",
      "Language": "1",
      "TypeOfDenomination": "001",
      "Denomination": "ASSOCIATION TURQUE DES EXPORTATEURS DE TEXTILE ET D'HABILLEMENT D'ISTANBUL - ITKIB"
    }
  ],
  "addresses": [
    {
      "_id": {
        "$oid": "6a6615a0e03570e5a541c88e"
      },
      "EntityNumber": "0257.883.408",
      "TypeOfAddress": "REGO",
      "CountryNL": "Turkije",
      "CountryFR": "Turquie",
      "Zipcode": "34196",
      "MunicipalityNL": "yenibosna - Istamboul",
      "MunicipalityFR": "yenibosna - Istamboul",
      "StreetNL": "itkib bis ticaret komplexi b/blok coban cesme mekvil sanayi/caddesi",
      "StreetFR": "itkib bis ticaret komplexi b/blok coban cesme mekvil sanayi/caddesi",
      "HouseNumber": "0",
      "Box": "",
      "ExtraAddressInfo": "",
      "DateStrikingOff": ""
    }
  ],
  "contacts": [],
  "activities": [],
  "establishments": [
    {
      "_id": {
        "$oid": "6a66153fe03570e5a5f6e057"
      },
      "EstablishmentNumber": "2.076.372.003",
      "StartDate": "02-05-1996",
      "EnterpriseNumber": "0257.883.408",
      "denominations": [
        {
          "_id": {
            "$oid": "6a66158ae03570e5a52f51b1"
          },
          "EntityNumber": "2.076.372.003",
          "Language": "1",
          "TypeOfDenomination": "003",
          "Denomination": "ITKIB"
        }
      ],
      "addresses": [
        {
          "_id": {
            "$oid": "6a6615d4e03570e5a5561a3f"
          },
          "EntityNumber": "2.076.372.003",
          "TypeOfAddress": "BAET",
          "CountryNL": "",
          "CountryFR": "",
          "Zipcode": "1040",
          "MunicipalityNL": "Brussel",
          "MunicipalityFR": "Bruxelles",
          "StreetNL": "Wetstraat",
          "StreetFR": "Rue de la Loi",
          "HouseNumber": "28",
          "Box": "",
          "ExtraAddressInfo": "",
          "DateStrikingOff": ""
        }
      ],
      "contacts": [],
      "activities": [
        {
          "_id": {
            "$oid": "6a6616d5e03570e5a5008e6a"
          },
          "EntityNumber": "2.076.372.003",
          "ActivityGroup": "003",
          "NaceVersion": "2008",
          "NaceCode": "63910",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6616d5e03570e5a5008e6b"
          },
          "EntityNumber": "2.076.372.003",
          "ActivityGroup": "003",
          "NaceVersion": "2003",
          "NaceCode": "92400",
          "Classification": "MAIN"
        },
        {
          "_id": {
            "$oid": "6a6616d5e03570e5a5008e6c"
          },
          "EntityNumber": "2.076.372.003",
          "ActivityGroup": "003",
          "NaceVersion": "2025",
          "NaceCode": "60310",
          "Classification": "MAIN"
        }
      ]
    }
  ],
  "branches": [
    {
      "_id": {
        "$oid": "6a66155de03570e5a50e6862"
      },
      "Id": "9.000.006.626",
      "StartDate": "01-09-1995",
      "EnterpriseNumber": "0257.883.408",
      "denominations": [],
      "addresses": [
        {
          "_id": {
            "$oid": "6a66160ce03570e5a56da24a"
          },
          "EntityNumber": "9.000.006.626",
          "TypeOfAddress": "ABBR",
          "CountryNL": "",
          "CountryFR": "",
          "Zipcode": "1040",
          "MunicipalityNL": "Brussel",
          "MunicipalityFR": "Bruxelles",
          "StreetNL": "Wetstraat",
          "StreetFR": "Rue de la Loi",
          "HouseNumber": "28",
          "Box": "",
          "ExtraAddressInfo": "",
          "DateStrikingOff": ""
        }
      ],
      "contacts": [],
      "activities": []
    }
  ]
}


{'_id': '0257.883.408',
 'EnterpriseNumber': '0257.883.408',
 'Status': 'AC',
 'JuridicalSituation': '000',
 'TypeOfEnterprise': '2',
 'JuridicalForm': '030',
 'JuridicalFormCAC': '',
 'StartDate': '01-09-1995',
 'denominations': [{'_id': {'$oid': '6a66155ee03570e5a50e9ef2'},
   'EntityNumber': '0257.883.408',
   'Language': '1',
   'TypeOfDenomination': '001',
   'Denomination': "ASSOCIATION TURQUE DES EXPORTATEURS DE TEXTILE ET D'HABILLEMENT D'ISTANBUL - ITKIB"}],
 'addresses': [{'_id': {'$oid': '6a6615a0e03570e5a541c88e'},
   'EntityNumber': '0257.883.408',
   'TypeOfAddress': 'REGO',
   'CountryNL': 'Turkije',
   'CountryFR': 'Turquie',
   'Zipcode': '34196',
   'MunicipalityNL': 'yenibosna - Istamboul',
   'MunicipalityFR': 'yenibosna - Istamboul',
   'StreetNL': 'itkib bis ticaret komplexi b/blok coban cesme mekvil sanayi/caddesi',
   'StreetFR': 'itkib bis ticaret komplexi b/blok coban cesme mekvil sanayi/caddesi',
   'HouseNumber': '0',
   'Box': '',
   'ExtraAddressInfo': '',
